# 04 — Entraînement Cloud (Google Colab GPU)
**NightVision Gen — Entraînement DDPM conditionnel depuis zéro (Modèles A, B, C)**

Ce notebook est conçu pour être exécuté sur **Google Colab avec accélérateur GPU**.
Il récupère le code depuis le dépôt GitHub, sauvegarde les checkpoints sur **Google Drive** de manière persistante, et permet de sélectionner la variante à entraîner sans toucher au code source.

---
### Plan d'exécution :
1. Vérification de l'environnement GPU (NVIDIA T4 / V100 / A100).
2. Montage de Google Drive pour la persistance des poids.
3. Clonage du dépôt GitHub et installation des dépendances.
4. Téléchargement ou accès aux données LOL-v2.
5. Sélection de la variante : **Modèle A**, **Modèle B** ou **Modèle C**.
6. Initialisation aléatoire du U-Net (entraînement depuis zéro).
7. Lancement de l'entraînement avec le `Trainer` modulaire.
8. Visualisation des courbes de perte et des exemples de validation.
9. Sauvegarde et reprise fiable après déconnexion.

In [ ]:
# 1. Vérification de l'environnement GPU
!nvidia-smi
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté :", torch.cuda.get_device_name(0))
    print("Mémoire VRAM :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "Go")
else:
    print("⚠️ Aucun GPU détecté. Activez l'accélérateur GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 2. Montage de Google Drive pour la persistance des checkpoints
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
GDRIVE_DIR = Path('/content/drive/MyDrive/NightVision_Gen')
GDRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("✅ Google Drive monté avec succès :", GDRIVE_DIR)

In [ ]:
# 3. Clone du dépôt GitHub et installation du package
# Remplacez l'URL par l'URL de votre dépôt GitHub
GITHUB_REPO_URL = "https://github.com/HASSANDADDA-04/nightvision-gen.git"

%cd /content
if not os.path.exists("nightvision-gen"):
    !git clone $GITHUB_REPO_URL
%cd /content/nightvision-gen

# Installation des dépendances et du package en mode éditable
!pip install -r requirements-colab.txt
!pip install -e . --no-build-isolation
print("✅ Package nightvision installé avec succès sur Colab.")

In [ ]:
# 4. Configuration des chemins et accès au dataset LOL-v2
import sys
sys.path.append(str(Path("/content/nightvision-gen/src").resolve()))

# Création des liens symboliques vers Google Drive pour les checkpoints et datasets
!mkdir -p artifacts/checkpoints artifacts/logs artifacts/generated_images
!mkdir -p data/raw/LOLv2

# Si LOL-v2 est déjà sur votre Drive, faites un lien symbolique :
# !cp -r /content/drive/MyDrive/NightVision_Gen/data/LOLv2/* data/raw/LOLv2/

print("Vérification des chemins d'accès effectuée.")

In [ ]:
# 5. Sélection de la variante du modèle à entraîner
# @title Configuration de l'Expérience { run: "auto" }
MODEL_VARIANT = "C" # @param ["A", "B", "C"]
EPOCHS = 100 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
LEARNING_RATE = 1e-4 # @param {type:"number"}
LAMBDA_EDGE = 0.05 # @param {type:"number"}
RESUME_TRAINING = False # @param {type:"boolean"}

print(f"🎯 Variante sélectionnée : Modèle {MODEL_VARIANT}")
print(f"   Époques : {EPOCHS} | Batch size : {BATCH_SIZE} | LR : {LEARNING_RATE}")
if MODEL_VARIANT == "C":
    print(f"   Poids de la perte de contours (lambda_edge) : {LAMBDA_EDGE}")

In [ ]:
# 6. Chargement de la configuration YAML correspondante
from nightvision.utils.config import load_config

config_file = f"configs/train_{MODEL_VARIANT}.yaml"
cfg = load_config(config_file)

# Mise à jour avec les paramètres du notebook
cfg["training"]["epochs"] = EPOCHS
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["training"]["learning_rate"] = LEARNING_RATE
if MODEL_VARIANT == "C":
    cfg["loss"]["lambda_edge"] = LAMBDA_EDGE

# Redirection de la sauvegarde vers Google Drive pour éviter toute perte en cas de déconnexion
drive_ckpt_dir = GDRIVE_DIR / f"checkpoints/model_{MODEL_VARIANT}"
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["checkpoint_dir"] = str(drive_ckpt_dir)

print("Configuration prête. Sauvegarde des checkpoints ciblée sur :", drive_ckpt_dir)

In [ ]:
# 7. Préparation des DataLoaders LOL-v2
from torch.utils.data import DataLoader
from nightvision.data.split_manager import LOLv2SplitManager
from nightvision.data.dataset import LOLv2PatchDataset, SyntheticPairDataset

compute_retinex = MODEL_VARIANT in ("B", "C")
manifest_path = Path("data/processed/lol_v2_splits.json")

if manifest_path.exists():
    splits = LOLv2SplitManager.load_split_manifest(manifest_path)
    train_ds = LOLv2PatchDataset(splits["train"], compute_retinex=compute_retinex)
    val_ds = LOLv2PatchDataset(splits["val"], compute_retinex=compute_retinex)
else:
    print("Manifeste non trouvé, initialisation du split manager...")
    lol_mgr = LOLv2SplitManager(root_dir="data/raw/LOLv2", seed=42)
    splits = lol_mgr.create_splits()
    if len(splits["train"]) > 0:
        train_ds = LOLv2PatchDataset(splits["train"], compute_retinex=compute_retinex)
        val_ds = LOLv2PatchDataset(splits["val"], compute_retinex=compute_retinex)
    else:
        print("⚠️ Images LOL-v2 introuvables. Utilisation du dataset synthétique pour démonstration.")
        train_ds = SyntheticPairDataset(32, compute_retinex=compute_retinex)
        val_ds = SyntheticPairDataset(8, compute_retinex=compute_retinex)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders prêts : {len(train_loader)} batches d'entraînement.")

In [ ]:
# 8. Initialisation du Trainer et Lancement de l'Entraînement
from nightvision.training.trainer import Trainer

trainer = Trainer(config=cfg, train_loader=train_loader, val_loader=val_loader)

# Reprise éventuelle depuis un checkpoint existant
latest_ckpt = drive_ckpt_dir / "checkpoint_latest.pt"
if RESUME_TRAINING and latest_ckpt.exists():
    print("Reprise de l'entraînement depuis :", latest_ckpt)
    trainer.resume_from_checkpoint(latest_ckpt)

# Lancement de l'entraînement
print(f"🚀 Démarrage de l'entraînement du Modèle {MODEL_VARIANT} depuis zéro...")
trainer.train()

In [ ]:
# 9. Visualisation des Courbes d'Apprentissage
import pandas as pd
import matplotlib.pyplot as plt

history_file = list(Path(cfg["paths"]["log_dir"]).glob("*_metrics.csv"))
if history_file:
    df = pd.read_csv(history_file[0])
    plt.figure(figsize=(10, 4))
    plt.plot(df["epoch"], df["train_loss"], label="Perte Entraînement", color="#3b82f6", lw=2)
    if "val_loss" in df.columns:
        plt.plot(df["epoch"], df["val_loss"], label="Perte Validation", color="#ef4444", lw=2)
    plt.xlabel("Époque")
    plt.ylabel("Perte")
    plt.title(f"Historique d'Entraînement — Modèle {MODEL_VARIANT}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.show()
else:
    print("Fichier de métriques en attente de génération.")

In [ ]:
# 10. Copie de Sauvegarde finale sur Google Drive
!cp -r artifacts/logs/* {str(GDRIVE_DIR)}/logs/ || true
!cp -r artifacts/generated_images/* {str(GDRIVE_DIR)}/generated_images/ || true
print("✅ Checkpoints et résultats synchronisés avec succès sur Google Drive.")